<a href="https://colab.research.google.com/github/oskmaimloe-jp/rag_test_20261007/blob/main/chapter2/openai_api_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 0: ハンズオンの準備
---

## API キーの設定
*  左ナビゲーションで [**シークレット**] アイコン (鍵形のアイコン) をクリックします。
*  [**新しいシークレットを追加**] をクリックし、[**名前**] に `OPENAI_API_KEY` と入力し、その [**値**] に指定されたキーを入力します。
*  設定したシークレットの [**ノートブックからのアクセス**] を有効にします。
*  入力が完了したら、下のセルを実行します。

In [3]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Section 1: トークン
---

## Tokenizerとtiktoken
- LLM などの自然言語処理を行う機械学習モデルは、テキストをトークンという単位で処理する。
- テキストデータなどをトークン化 (tokenize) するプログラムを **Tokenizer** という。
- トークン化の仕方や入力可能なトークン数は LLM によって異なり、入力するトークン数に応じて API 利用料金が課金される。
- **tiktoken** は OSS の高速な Tokenizer である。

In [ ]:
# !pip -q install tiktoken

In [4]:
!pip freeze | grep tiktoken

tiktoken==0.14.0


In [5]:
import tiktoken

text = "It’s easy to make something cool with LLMs, but very hard to make something production-ready with them."

encoding = tiktoken.encoding_for_model("gpt-4o-mini")
tokens = encoding.encode(text)
print(len(tokens))

23


In [7]:
text = "意味が同じ文章でも日本語の方がトークン数が多くなりがち、コストが上がりがち"

encoding = tiktoken.encoding_for_model("gpt-4o-mini")
tokens = encoding.encode(text)
print(len(tokens))

31


In [8]:
text = "LLMを使ってクールなものを作るのは簡単だが、プロダクションで使えるものを作るのは非常に難しい。"

encoding = tiktoken.encoding_for_model("gpt-4o-mini")
tokens = encoding.encode(text)
print(len(tokens))

37


# Section 2: OpenAI API の利用
---

### プロンプト設計のためのサイト
- [OpenAI Platform](https://platform.openai.com/playground/chat?models=gpt-4o-mini)
- [Google AI Studio](https://aistudio.google.com/)
- [Anthropic Console](https://console.anthropic.com/)

## Chat Completions API

In [ ]:
# !pip install openai

In [9]:
!pip freeze | grep openai

openai==2.54.0


### Chat Completions APIの呼び出し

In [15]:
from openai import OpenAI
client = OpenAI()

completion = client.chat.completions.create(
  model="gpt-6.1-sol",
  messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Do you know John?"}
  ]
)

print(completion.choices[0].message)

ChatCompletionMessage(content='Yes—LangChain is a framework for building applications powered by large language models (LLMs). It helps connect models to prompts, data sources, tools, and workflows.\n\nCommon uses include:\n- **RAG:** Answering questions using your documents or databases.\n- **Agents:** Letting models call tools to complete tasks.\n- **Chat applications:** Managing conversation context and structured outputs.\n\nIts ecosystem includes **LangGraph** for stateful agent workflows and **LangSmith** for tracing and evaluation.\n\nAre you looking for an overview, help with code, or advice on a specific project?', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


### 会話履歴を踏まえた応答を得る

In [14]:
completion = client.chat.completions.create(
  model="gpt-6.1-sol",
  messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello! I'm John."},
        {"role": "assistant", "content": "Hello John! How can I assist you today?"},
        {"role": "user", "content": "Do you know my name?"}
    ]
)

print(completion.choices[0].message)

ChatCompletionMessage(content='Yes—your name is John.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)


### ストリーミングで応答を得る

In [25]:
stream = client.chat.completions.create(
  model="gpt-4o-mini",
  messages=[
        {"role": "system", "content": "あなたは AI や機械学習に精通したプロフェッショナルです。"},
        {"role": "user", "content": "AI とは何ですか。"}
  ],
  stream=True
)
#print(stream.choices[0].message)

for chunk in stream:
    if chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")

AI（人工知能）とは、人間の知能を模倣またはシミュレーションするために開発されたコンピュータシステムやプログラムのことを指します。AIは、学習（機械学習）、推論、問題解決、理解（自然言語処理）、認識（画像認識など）など、人間が行う様々な知的作業を自動化することを目的としています。

AIは大きく分けて以下の2つのカテゴリーに分類されます：

1. **狭義のAI（Weak AI）**: 特定のタスクや問題に特化したAIです。例としては、音声認識やチェスのプレイ、画像認識などがあります。

2. **汎用AI（Strong AI）**: 人間のように多様なタスクをこなす能力を持ち、自己学習や自己改善ができるAIですが、現在の技術では実現されていません。

AIの技術は、データの収集・分析、アルゴリズムの設計、計算資源の利用などに依存しており、さまざまな分野（医療、金融、交通、エンターテインメントなど）で応用されています。

# Section 3: Function calling
---
Function Calling に対応しているモデルでは、利用可能な関数の情報をモデルに渡すことで、モデルに必要に応じた関数の利用を選択させることができる。

(※ 以下では [OpenAI の公式ドキュメント](https://platform.openai.com/docs/guides/function-calling) をもとに一部改変したコードを使用している)

## 関数を用意する  
天気予報の情報を出力する関数  
(実際に外部の天気予報サービスを利用するのではなく、既定の予報を返す疑似的な天気予報の関数)

In [26]:
import json

def get_current_weather(location, unit="celsius"):
    weather_info = {
        "location": location,
        "temperature": "25",
        "unit": "celsius",
        "forecast": ["sunny", "windy"],
    }
    return json.dumps(weather_info)

## 関数のリストを定義する
- 利用可能な関数のリストを定義する。  
- それぞれの関数については、辞書で関数名や関数についての説明、関数を呼び出す際のパラメータ等を定義する。  
- 下の例では、`tools` という名前のリストに要素として関数が 1 つだけ入っている。

In [27]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. Tokyo",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

## 関数のリストを渡して LLM を呼び出す

In [28]:
messages = [{"role": "user", "content": "What's the weather like in Tokyo?"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",  # auto is default, but we'll be explicit
)

print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Pt2ZsOUplNWuYESX9mvlhjCD', function=Function(arguments='{"location":"Tokyo"}', name='get_current_weather'), type='function')])


In [29]:
print(response.choices[0].message.tool_calls)

[ChatCompletionMessageFunctionToolCall(id='call_Pt2ZsOUplNWuYESX9mvlhjCD', function=Function(arguments='{"location":"Tokyo"}', name='get_current_weather'), type='function')]


In [30]:
print(response.choices[0].message.tool_calls[0].function.name)
print(response.choices[0].message.tool_calls[0].function.arguments)

get_current_weather
{"location":"Tokyo"}


- LLM が関数を使用することを選択した場合、使用する関数の名前や関数を呼び出す際のパラメータを回答する。
- LLM は `tools` で定義されている関数の `name` や `description` 、`parameters` を参照して回答を生成する。

## 関数を実行する  
- 上記の LLM の回答から関数名やパラメータを取得し、関数を実行する

In [31]:
response_message = response.choices[0].message
tool_call = response_message.tool_calls[0]

available_functions = {
    "get_current_weather": get_current_weather,
}
messages.append(response_message)
function_name = tool_call.function.name
function_to_call = available_functions[function_name]
function_args = json.loads(tool_call.function.arguments)
function_response = function_to_call(
    location=function_args.get("location"),
    unit=function_args.get("unit"),
)

print(function_response)

{"location": "Tokyo", "temperature": "25", "unit": "celsius", "forecast": ["sunny", "windy"]}


## 関数の実行結果を `messages` に追加する

In [32]:
messages.append({
    "tool_call_id": tool_call.id,
    "role": "tool",
    "name": function_name,
    "content": function_response,
})

## `messages` を渡して LLM を再度呼び出す

In [33]:
second_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
)

print(second_response.choices[0].message)

ChatCompletionMessage(content='The weather in Tokyo is currently 25°C and it is sunny and windy.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)
